# 🚀 Prompting Techniques Lab with LangChain

**Objective:** Master the art of prompt engineering through hands-on practice with LangChain

In this lab, you'll learn:
- How to structure effective prompts
- Core prompting techniques that dramatically improve AI outputs
- How to use LangChain to implement professional prompting strategies
- Real-world applications of prompt engineering

---


In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv
load_dotenv()

# Initialize the language model
llm = init_chat_model("gpt-4o-mini", model_provider="openai")

print("✅ Setup complete!")


---

## 🎯 Section 1: The Anatomy of a Good Prompt

A good prompt has four key elements:
1. **Context** - Background information
2. **Task** - What you want the AI to do
3. **Format** - How to structure the output
4. **Constraints** - Boundaries and requirements

Let's see the difference between weak and strong prompts.


In [ ]:
# WEAK PROMPT - Vague and unstructured
weak_prompt = "Write about marketing for a coffee shop"

response = llm.invoke(weak_prompt)
print("❌ WEAK PROMPT OUTPUT:")
print("=" * 50)
print(response.content)
print("\n" + "=" * 50)


In [ ]:
# STRONG PROMPT - Structured with all four elements
strong_prompt = """
Context: I'm a small business owner launching a new coffee shop in downtown Seattle. 
My target audience is young professionals and students aged 22-35.

Task: Create a social media marketing strategy focused on Instagram and TikTok.

Format: Provide your response in 3 paragraphs:
1. Content strategy and themes
2. Specific content ideas (at least 5)
3. Posting frequency and best practices

Constraints:
- Keep the tone casual and energetic
- Focus on budget-friendly tactics
- Include trending content formats
"""

response = llm.invoke(strong_prompt)
print("✅ STRONG PROMPT OUTPUT:")
print("=" * 50)
print(response.content)
print("\n" + "=" * 50)


### Prompt Template

In [ ]:
from langchain_core.prompts import PromptTemplate

# STRONG PROMPT - Structured with all four elements using a prompt template
strong_prompt_template = PromptTemplate(
    input_variables=["business_type", "location", "target_audience", "channels"],
    template="""
Context: I'm a small business owner launching a new {business_type} in {location}. 
My target audience is {target_audience}.

Task: Create a social media marketing strategy focused on {channels}.

Format: Provide your response in 3 paragraphs:
1. Content strategy and themes
2. Specific content ideas (at least 5)
3. Posting frequency and best practices

Constraints:
- Keep the tone casual and energetic
- Focus on budget-friendly tactics
- Include trending content formats
"""
)


chain = strong_prompt_template | llm | StrOutputParser()
prompt_vars = {
    "business_type": "coffee shop",
    "location": "downtown Seattle",
    "target_audience": "young professionals and students aged 22-35",
    "channels": "Instagram and TikTok"
}

response = chain.invoke(prompt_vars)
print(response)

In [ ]:
prompt_vars = {
    "business_type": "tiffin center",
    "location": "hyderabad",
    "target_audience": "students and working professionals",
    "channels": "Instagram and Facebook"
}

response = chain.invoke(prompt_vars)
print(response)


## 🎭 Section 2: Technique 1 - Role Assignment
Assigning a role helps the AI frame responses with appropriate expertise and perspective.

You are an experienced fitness coach with 10 years of experience. Respond with the expertise, tone, and perspective appropriate to this role

I want to lose 10 pounds in 2 months. I work a desk job and have never worked out consistently. What should I do?

---
You are a Doctor with 10 years of experience. Respond with the expertise, tone, and perspective appropriate to this role
I want to lose 10 pounds in 2 months. I work a desk job and have never worked out consistently. What should I do?


In [ ]:
# Using LangChain's ChatPromptTemplate for role assignment
role_template = ChatPromptTemplate.from_messages([
    ("system", "You are an experienced fitness coach with 10 years of experience. Respond with the expertise, tone, and perspective appropriate to this role."),
    ("human", "{user_input}")
])

chain = role_template | llm | StrOutputParser()

# Example 1: Fitness Coach
print("🏋️ FITNESS COACH PERSPECTIVE:")
print("=" * 50)
response = chain.invoke({
    "user_input": "I want to lose 10 pounds in 2 months. I work a desk job and have never worked out consistently. What should I do?"
})
print(response)
print("\n")



In [ ]:

# Example 2: Doctor
role_template = ChatPromptTemplate.from_messages([
    ("system", "You are a Doctor with 10 years of experience. Respond with the expertise, tone, and perspective appropriate to this role."),
    ("human", "{user_input}")
])

chain = role_template | llm | StrOutputParser()
print("🥗 DOCTOR PERSPECTIVE:")
print("=" * 50)
response = chain.invoke({
    "user_input": "I want to lose 10 pounds in 2 months. I work a desk job and have never worked out consistently. What should I do?"
})
print(response)

In [ ]:
# Real-world example: Code review from different perspectives

code_to_review = open("sample.java", "r").read()

roles_and_system_msgs = [
    (
        "Senior Java Developer",
        "You are a senior Java developer focused on code quality and best practices. Respond with the expertise, tone, and perspective appropriate to this role."
    ),
    (
        "Performance Optimization Expert",
        "You are a performance optimization expert. Respond with the expertise, tone, and perspective appropriate to this role."
    ),
    (
        "Security Engineer",
        "You are a security engineer looking for potential vulnerabilities. Respond with the expertise, tone, and perspective appropriate to this role."
    )
]

for role, system_msg in roles_and_system_msgs:
    print(f"\n{'=' * 60}")
    print(f"📋 REVIEW FROM: {role.upper()}")
    print("=" * 60)
    static_role_template = ChatPromptTemplate.from_messages([
        ("system", system_msg),
        ("human", "{user_input}")
    ])
    static_chain = static_role_template | llm | StrOutputParser()
    response = static_chain.invoke({
        "user_input": f"Review this Java function and provide specific feedback:\n\n{code_to_review}"
    })
    print(response)


### 🔥 Exercise 2: Role Assignment

Ask for investment advice from two different roles: a conservative financial advisor vs. a cryptocurrency enthusiast.


In [ ]:
# YOUR CODE HERE
investment_question = "I have $10,000 to invest. I'm 28 years old and willing to take some risk. What should I do?"

# Try with different roles
# Role 1: Conservative financial advisor
# Role 2: Crypto enthusiast


---

## 📚 Section 3: Technique 2 - Few-Shot Prompting

Few-shot prompting provides examples to teach the AI your desired format, style, or pattern.


Write engaging product descriptions for the following products


Noise-Cancelling Headphones
Standing Desk
4K Webcam

Write engaging product descriptions for the following products

Examples for your reference:
Product : Wireless Mouse
Description : 🖱️ **Glide Through Your Workday** - Experience butter-smooth precision with our ergonomic wireless mouse. 18-month battery life means you'll forget it even needs power. *Your wrist will thank you.*

Product: Mechanical Keyboard
Description: ⌨️ **Type Like You Mean It** - Every keystroke is a satisfying click of productivity. RGB lighting that adapts to your mood. Built to survive coffee spills and late-night coding marathons. *This is your keyboard's final form.*

Provide the description in the same format as above

Noise-Cancelling Headphones
Standing Desk
4K Webcam

Instruction:
Identify the pattern in the sequence and provide the next item.

Examples:
Sequence: 3, 6, 12, 24
Next: 48

Sequence: B, E, H, K
Next: N

Task:
Sequence: 10, 20, 40, 80
Next:

In [ ]:
# Example : Product descriptions with consistent style
print("\n🛍️ CONSISTENT PRODUCT DESCRIPTIONS:")
print("=" * 60)

product_examples = [
    {
        "product": "Wireless Mouse",
        "description": "🖱️ **Glide Through Your Workday** - Experience butter-smooth precision with our ergonomic wireless mouse. 18-month battery life means you'll forget it even needs power. *Your wrist will thank you.*"
    },
    {
        "product": "Mechanical Keyboard",
        "description": "⌨️ **Type Like You Mean It** - Every keystroke is a satisfying click of productivity. RGB lighting that adapts to your mood. Built to survive coffee spills and late-night coding marathons. *This is your keyboard's final form.*"
    }
]

product_example_template = """
Product: {product}
Description: {description}
"""

product_example_prompt = PromptTemplate(
    input_variables=["product", "description"],
    template=product_example_template
)

product_few_shot = FewShotPromptTemplate(
    examples=product_examples,
    example_prompt=product_example_prompt,
    prefix="Write engaging product descriptions in this exact style:\n",
    suffix="\nProduct: {product}\nDescription:",
    input_variables=["product"]
)



product_chain = product_few_shot | llm | StrOutputParser()

# Generate descriptions for new products
new_products = ["Noise-Cancelling Headphones", "Standing Desk", "4K Webcam"]

for product in new_products:
    description = product_chain.invoke({"product": product})
    print("Product : ",product)
    print(f"\n{description}")
    print("-" * 60)

In [ ]:
# Example: Converting casual language to professional business language
examples = [
    {
        "casual": "Hey, can you send me that report?",
        "professional": "Could you please share the report at your earliest convenience?"
    },
    {
        "casual": "The meeting was a total mess.",
        "professional": "The meeting could have been better organized."
    },
    {
        "casual": "This project is taking forever!",
        "professional": "The project timeline has extended beyond initial estimates."
    }
]

# Create few-shot prompt template
example_template = """
Casual: {casual}
Professional: {professional}
"""

example_prompt = PromptTemplate(
    input_variables=["casual", "professional"],
    template=example_template
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="Convert casual sentences into professional business language:\n",
    suffix="\nCasual: {input}\nProfessional:",
    input_variables=["input"]
)

chain = few_shot_prompt | llm | StrOutputParser()

# Test it
test_sentences = [
    "I'm not sure about this idea.",
    "Can we talk about this later? I'm swamped.",
    "That's a terrible idea and it won't work."
]

print("🎯 FEW-SHOT LEARNING IN ACTION:")
print("=" * 50)
for sentence in test_sentences:
    result = chain.invoke({"input": sentence})
    print(f"\nCasual: {sentence}")
    print(f"Professional: {result}")
    print("-" * 50)


### 🔥 Exercise 3: Create Your Few-Shot Prompt

Create a few-shot prompt that converts technical jargon into simple explanations.


In [ ]:
# YOUR CODE HERE
# Create examples that convert technical terms to simple language
# Example: "API" -> "A way for different software applications to talk to each other"

jargon_examples = [
    # Add your examples here
]

# Build and test your few-shot prompt


---

## 🧠 Section 4: Technique 3 - Chain-of-Thought (Step-by-Step Reasoning)

For complex tasks, asking the AI to think step-by-step leads to more accurate results.


In [ ]:
# Example 1: Business decision making
decision_template = ChatPromptTemplate.from_messages([
    ("system", "You are a business consultant. Always break down complex decisions into clear steps."),
    ("human", """
    Analyze this situation using the following framework:
    
    1. **Identify the key factors**
    2. **List pros and cons for each option**
    3. **Consider short-term vs long-term implications**
    4. **Assess risks**
    5. **Provide your recommendation with reasoning**
    
    Situation: {situation}
    """)
])

decision_chain = decision_template | llm | StrOutputParser()

situation = """
I run a small software company with 10 employees. We've been offered two opportunities:

Option A: A large enterprise client wants us to build a custom solution for $200K, 
but it would require 6 months of dedicated work and we'd need to hire 3 more developers.

Option B: Keep focusing on our SaaS product that currently has 500 paying customers 
at $50/month each. We're growing 10% month-over-month.

What should I do?
"""

response = decision_chain.invoke({"situation": situation})
print("\n💼 CHAIN-OF-THOUGHT: Business Decision")
print("=" * 60)
print(response)


In [ ]:
# Example 2: Debugging code with step-by-step analysis
debug_prompt = f"""
Analyze this code bug using step-by-step reasoning:

```python
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)

# This crashes sometimes
scores = []
average = calculate_average(scores)
print(f"Average: {{average}}")
```

Please:
1. Identify what happens when this code runs
2. Explain why it crashes
3. Describe the edge cases
4. Provide a fixed version with error handling
5. Suggest additional improvements


For each step provide you thoughts and reasoning.

The output should be in the following format:
{{
    "step_1": {{
        "thought": "...",
        "reasoning": "...",
        "output": "..."
    }}
}}
"""

response = llm.invoke(debug_prompt)
print("\n🐛 CHAIN-OF-THOUGHT: Debugging")
print("=" * 60)
print(response.content)


### 🔥 Exercise 4: Chain-of-Thought Problem Solving

Create a step-by-step prompt to help someone decide whether to buy or rent a home.


In [ ]:
# YOUR CODE HERE
# Create a prompt that uses chain-of-thought to analyze the buy vs rent decision
# Include specific financial details and ask for step-by-step analysis


---

## 📋 Section 5: Technique 4 - Output Format Control

Specifying the exact format ensures you get usable, structured output.


In [ ]:
# Example 1: Structured JSON output
print("\n📊 STRUCTURED JSON OUTPUT:")
print("=" * 60)

json_template = ChatPromptTemplate.from_messages([
    ("system", "You are a customer feedback analyst. Always respond with valid JSON only."),
    ("human", """
Analyze the following customer review and provide your analysis in this EXACT JSON format:

{{
    "sentiment": "positive/negative/neutral",
    "sentiment_score": 1-10,
    "main_issues": ["issue1", "issue2"],
    "main_praises": ["praise1", "praise2"],
    "urgency": "low/medium/high",
    "suggested_response": "brief response text"
}}

Review: {review}

Provide ONLY the JSON, no additional text.
    """)
])

json_chain = json_template | llm 

review_text = """I've been using this project management tool for 3 months. The interface is 
beautiful and intuitive, which my team loves. However, the mobile app crashes constantly 
and we're missing critical notifications. For the price we're paying ($50/user/month), 
this is unacceptable. We're considering switching to a competitor if this isn't fixed soon."""

response = json_chain.invoke({"review": review_text})
print(response)

print("📊 STRUCTURED JSON OUTPUT:")
print("=" * 60)
print(response.content)

data = response.content
# # You can parse this JSON
# import json
# try:
#     data = json.loads(response.content) #Converting the string to a dictionary(json)

# except:
#     print("\n⚠️ Response wasn't pure JSON, but that's okay for demo purposes")

print(data["sentiment"])

In [ ]:
# Example 2: Markdown table format
table_prompt = """
Compare Python, JavaScript, and Java for web development.

Provide your answer in a Markdown table with these exact columns:
| Language | Primary Use Case | Learning Curve | Performance | Community Size | Best For |

Fill in each cell with concise information (max 10 words per cell).
"""

response = llm.invoke(table_prompt)
print("\n📊 MARKDOWN TABLE OUTPUT:")
print("=" * 60)
print(response.content)


In [ ]:
# Example 3: Bullet points with specific structure
bullet_prompt_template = PromptTemplate(
    input_variables=["number", "topic"],
    template="""
Provide {number} tips for {topic}.

Format each tip as:
🎯 **[Number]. [Title in 3-5 words]**
   - What: [One sentence explanation]
   - Why: [One sentence benefit]
   - Example: [Concrete example]

Make it actionable and specific.
""")

chain = bullet_prompt_template | llm | StrOutputParser()
response = chain.invoke({
    "number": "5",
    "topic": "improving code review quality in a development team"
})

print("\n📝 STRUCTURED BULLET POINTS:")
print("=" * 60)
print(response)


### 🔥 Exercise 5: Control the Output Format

Create a prompt that generates a weekly meal plan in a specific table format with calories and prep time.


In [ ]:
# YOUR CODE HERE
# Create a prompt that asks for a meal plan in a specific format
# Include columns: Day, Meal Type, Dish Name, Calories, Prep Time


---

## 🎯 Section 6:Prompt Chaining

Break complex tasks into a series of smaller prompts where each output feeds into the next.


In [ ]:
# Example: Multi-step content creation pipeline
# Step 1: Brainstorm ideas
brainstorm_prompt = """
I need to create content for a LinkedIn post.

Topic: How AI is changing software development
Target audience: Mid-level developers

Provide 5 unique angles or perspectives on this topic.
For each angle, write one sentence explaining why it would resonate with the audience.
"""

brainstorm_response = llm.invoke(brainstorm_prompt)
print("🧠 STEP 1: BRAINSTORM")
print("=" * 60)
print(brainstorm_response.content)
print("\n")

# Step 2: Expand on chosen angle
expand_prompt = f"""
Based on these brainstormed ideas:
{brainstorm_response.content}

Choose the most compelling angle and create a detailed outline for a LinkedIn post.
Include:
- Opening hook
- 3 main points with supporting details
- Personal insight or story
- Call to action
"""

expand_response = llm.invoke(expand_prompt)
print("📋 STEP 2: OUTLINE")
print("=" * 60)
print(expand_response.content)
print("\n")

# Step 3: Write the full post
write_prompt = f"""
Using this outline:
{expand_response.content}

Write a complete LinkedIn post (300-400 words).

Style guidelines:
- Start with a bold statement or question
- Use short paragraphs (2-3 sentences max)
- Include relevant emojis sparingly
- End with an engaging question to drive comments
- Professional yet personable tone
"""

final_post = llm.invoke(write_prompt)
print("✍️ STEP 3: FINAL POST")
print("=" * 60)
print(final_post.content)


---

## 🚀 Section 7: Real-World Use Cases

Let's apply everything we've learned to solve real problems.

### Use Case 1: Email Communication


In [ ]:
email_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional communication expert."),
    ("human", """
    Write a professional email with the following details:
    
    Situation: {situation}
    Recipient: {recipient}
    Goal: {goal}
    Tone: {tone}
    Key points to include: {key_points}
    
    Format:
    - Clear subject line
    - Professional greeting
    - 2-3 short paragraphs
    - Appropriate closing
    
    Length: {max_words} words maximum
    """)
])

email_chain = email_template | llm | StrOutputParser()

# Example: Project delay notification
email_result = email_chain.invoke({
    "situation": "Our development team encountered unexpected technical challenges with the authentication system",
    "recipient": "Client stakeholder (non-technical)",
    "goal": "Inform about 2-week delay while maintaining trust and confidence",
    "tone": "Professional, transparent, and solution-focused",
    "key_points": "Reason for delay (without jargon), new timeline, steps we're taking, commitment to quality",
    "max_words": "200"
})

print("📧 PROFESSIONAL EMAIL EXAMPLE:")
print("=" * 60)
print(email_result)


---

## 📚 Key Takeaways & Best Practices

### The Prompting Framework

**Remember: C-T-F-C**
1. **Context** - Who, what, where, why
2. **Task** - Clear, specific action
3. **Format** - How to structure output
4. **Constraints** - Boundaries and requirements

### Core Techniques Summary

1. **Role Assignment** - Give the AI expertise and perspective
2. **Few-Shot Learning** - Show examples of what you want
3. **Chain-of-Thought** - Ask for step-by-step reasoning
4. **Format Control** - Specify exact output structure
5. **Iterative Refinement** - Start simple, improve based on results
6. **Prompt Chaining** - Break complex tasks into steps

### Best Practices

✅ **Do:**
- Be specific and detailed
- Use positive instructions ("do this" vs "don't do that")
- Iterate and refine
- Test with different examples
- Save successful prompts for reuse

❌ **Don't:**
- Be vague or assume context
- Overload one prompt with multiple tasks
- Accept first output without reviewing
- Forget to specify tone and style
- Skip examples when format matters

### Advanced Tips

1. **Use delimiters** - Separate sections with ```, ---, ###
2. **Specify length** - "In 100 words", "3 paragraphs"
3. **Request multiple options** - "Give me 3 versions"
4. **Ask for explanations** - "Explain your reasoning"
5. **Set temperature** - Lower (0.1-0.3) for factual, higher (0.7-0.9) for creative

---

## 🎓 Next Steps

1. **Practice Daily**: Use these techniques in your actual work
2. **Build a Library**: Save your best prompts
3. **Experiment**: Try combining techniques
4. **Share & Learn**: Compare prompts with colleagues
5. **Stay Updated**: Prompting strategies evolve with models

### Resources

- [LangChain Documentation](https://python.langchain.com/docs/get_started/introduction)
- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)
- [Anthropic Prompt Engineering](https://docs.anthropic.com/claude/docs/prompt-engineering)

---